# Dirección **perpendicular** contra loops: `Qwen/Qwen3.5-0.8B` + MATH-500 (Colab A100 o L4)

En la corrida anterior (`discover_math500_08b_colab.ipynb`) el parche quedó débil: todas las ediciones de 2 capas
dañaban demasiado el razonamiento normal (freno de NLL 0.05) y solo pasó una de 1 capa. Fijo: 10.7% vs base 2.7%;
vía MoI empató con una dirección al azar (15.3% vs 18.7%).

**Idea (Ishiki Labs, bloque 2):** antes de editar, quitarle a la dirección del loop la parte que el razonamiento sano
también usa:  `u_perp = u − V_k V_kᵀ u`, con `V_k` = las k direcciones principales de la salida de la capa en
tokens normales. Así la edición frena el loop sin pegarle a lo sano, y ediciones más fuertes pueden pasar el freno.

1. Reusa lo ya hecho: elicit, diagnóstico y outputs del base (mismos problemas y semillas) de la corrida anterior.
2. `healthy`: un pase por los tokens normales → direcciones sanas por capa (~2 min).
3. `screen`: k ∈ {0 (normal), 1, 4, 16} × 1–2 capas × 16/64/256/todas las neuronas × α 4–24 = **160 configs**.
   La mejor fila decide k; las 3 mejores con esa k pasan a dev.
4. Dev a **16k** (antes 8k cortaba la mitad) → congelar → test 150 a 16k → reporte, con comparación pareada contra
   el parche normal de la corrida anterior.

~40–50 min en A100 (~1.5–2 h en L4). Resultado: `MyDrive/wsi_runs/FINALES/discover_math500_perp_qwen35-08b.json`.
Al final el runtime se apaga solo.

In [ ]:
MODEL = "Qwen/Qwen3.5-0.8B"
RUN_NAME = "discover_math500_perp_qwen35-08b"
REF_RUN = "discover_math500_qwen35-08b"     # la corrida con la dirección normal (ya terminada)
REPO = "/content/wsi"
BASE = "/content/base"
W = "/content/work_perp"
REF = "/content/ref_plain"
S = f"{REPO}/scripts/discover_math500_perp.py"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm sympy antlr4-python3-runtime==4.11.1 hf_transfer
!pip install -q flash-linear-attention || echo "fla opcional"
import vllm, transformers; print("vllm", vllm.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, glob, shutil, json, subprocess, datetime
from google.colab import drive
drive.mount("/content/drive")
DRIVE = f"/content/drive/MyDrive/wsi_runs/{RUN_NAME}"
DRIVE_REF = f"/content/drive/MyDrive/wsi_runs/{REF_RUN}"
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
BASE_CACHE = "/content/drive/MyDrive/wsi_runs/_base_cache"
VLLM_CACHE = "/content/drive/MyDrive/wsi_runs/_vllm_cache"
for d in (DRIVE, FINALES, BASE_CACHE, VLLM_CACHE, W, REF):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(f"{DRIVE_REF}/outputs_test_fast.jsonl"), f"falta la corrida normal en {DRIVE_REF}"
for f in glob.glob(f"{DRIVE_REF}/*"):          # the plain run (files only)
    if os.path.isfile(f) and not os.path.exists(f"{REF}/{os.path.basename(f)}"):
        shutil.copy2(f, REF)
for f in glob.glob(f"{DRIVE}/*"):              # resume this run
    if os.path.isfile(f) and not os.path.exists(f"{W}/{os.path.basename(f)}"):
        shutil.copy2(f, W)
if os.path.isdir(f"{DRIVE}/plain_ref") and not os.path.isdir(f"{W}/plain_ref"):
    shutil.copytree(f"{DRIVE}/plain_ref", f"{W}/plain_ref")
os.environ["WSI_SYNC_DIR"] = DRIVE
os.environ["WSI_BASE_CACHE_DIR"] = BASE_CACHE
os.environ["VLLM_CACHE_ROOT"] = VLLM_CACHE
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print("ya hecho en Drive:", sorted(os.listdir(DRIVE)) or "nada")

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git pull -q
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert os.path.exists(S), "el repo no tiene discover_math500_perp.py: falta push"
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)

In [ ]:
import time
# a vLLM engine left over from an interrupted cell keeps the GPU busy and the next run hangs: kill it first
!nvidia-smi --query-compute-apps=pid --format=csv,noheader | xargs -r kill -9

def run_step(cmd):
    print("\n>>>", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    buf, last = b"", 0.0
    while True:
        ch = p.stdout.read1(4096)
        if not ch:
            break
        buf += ch
        *lines, buf = buf.replace(b"\r\n", b"\n").replace(b"\r", b"\n").split(b"\n")
        for l in lines:
            s = l.decode(errors="replace").strip()
            if not s or "Attempted to load weight" in s:
                continue
            if "it/s" in s or "s/it" in s:  # tqdm: one line every 30 s
                if time.time() - last < 30:
                    continue
                last = time.time()
            print(s, flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Step failed: {' '.join(cmd)} (if it says STOP, that is the loop gate, not a bug)")

run_step(["python", "-u", S, "setup", "--work", W, "--ref", REF])
run_step(["python", "-u", S, "healthy", "--work", W])
run_step(["python", "-u", S, "screen", "--work", W])
run_step(["python", "-u", S, "select", "--work", W])
run_step(["python", "-u", S, "freeze", "--work", W, "--git-commit", COMMIT])
run_step(["python", "-u", S, "test", "--work", W, "--ck-root", "/content/ck"])
run_step(["python", "-u", S, "report", "--work", W])

In [ ]:
src = f"{W}/results_discover_math500_blind.json"
if not os.path.exists(src):
    raise RuntimeError("No se generó results_discover_math500_blind.json; revisa el error arriba.")
res = json.load(open(src))
res["_final"] = {
    "complete": True,
    "model": MODEL,
    "commit": COMMIT,
    "notebook": "discover_math500_perp_08b_colab.ipynb",
    "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as f:
    json.dump(res, f, indent=1)
os.replace(dst + ".tmp", dst)
print("Guardado final en Drive:", dst)

In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(90)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")
